In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


# 最終モデル
notebookの09_model-tuningより
* 使用モデル:XGBClassifier
* 使う特徴量:
  >数値列 =["logFare","Family","has_cabin","Pclass"]
  >
>カテゴリ列 = ["Embarked","Sex","age_binning"]
  

In [2]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# データ読み込み

In [3]:
#"PassengerId"をインデックスに指定
train_csv = pd.read_csv("/kaggle/input/competitions/titanic/train.csv").set_index("PassengerId")
test_csv = pd.read_csv("/kaggle/input/competitions/titanic/test.csv").set_index("PassengerId")

y = train_csv.Survived
X = train_csv.drop("Survived", axis=1)

# 特徴量の作成

In [4]:
class TitanicFeatureEngineering(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self,X,y=None):
        X = X.copy()

        X["Title"] = X["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
        X["Title"] = X["Title"].replace(["Mlle","Ms"], "Miss")
        X["Title"] = X["Title"].replace("Mme", "Mrs")
        rare_titles = [
             "Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major",
            "Rev", "Sir", "Jonkheer", "Dona"
        ]
        X["Title"] = X["Title"].replace(rare_titles, "Rare")

        self.title_age_median = X.groupby("Title")["Age"].median()
        self.global_age_median = X["Age"].median()

        return self

    def transform(self,X):
        X_new = X.copy()

        X_new["Family"] = X_new["Parch"].astype("Int64") + X_new["SibSp"].astype("Int64")

        fare_per_person = X_new["Fare"] / (X_new["Family"]+1)
        X_new["logFare"] = np.log1p(fare_per_person)#1人分の運賃で計算

        X_new["has_cabin"] = X_new["Cabin"].notnull().astype(int)

        X_new["Title"] = X_new["Name"].str.extract(r" ([A-Za-z]+)\.",expand = False)
        X_new["Title"] = X_new["Title"].replace(["Mlle","Ms"],"Miss")
        X_new["Title"] = X_new["Title"].replace("Mme","Mrs")
        rare_titles = [
            "Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major",
            "Rev", "Sir", "Jonkheer", "Dona"
        ]
        X_new["Title"] = X_new["Title"].replace(rare_titles,"Rare")

        X_new["Age"] = X_new["Age"].fillna(X_new["Title"].map(self.title_age_median))
        X_new["Age"] = X_new["Age"].fillna(self.global_age_median) #未知の値の対処

        child = X_new["Age"]<12
        young = (X_new["Age"]>=12) & (X_new["Age"]<20)
        adult = (X_new["Age"]>=20) & (X_new["Age"]<60)
        senior = X_new["Age"]>=60

        age_selection = [child,young,adult,senior]
        name = ["child","young","adult","senior"]
        X_new["age_binning"] = np.select(age_selection,name,default ="unknown") #未知の値の対処

        return X_new
        

# Pipelineの作成

In [5]:
numerical_transformer = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy="median")) #中央値埋め
    ]
)

categorical_transformer = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output = False)) #未知の値の対処、密行列で返す
    ]
)

num_cols =["logFare","Family","has_cabin","Pclass"] #数値列
cat_cols = ["Embarked","Sex","age_binning"] #カテゴリ列

preprocessor = ColumnTransformer(
    transformers = [
        ("num", numerical_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

# XGBClassifierモデルの設定
パラメータは、optunaを使用し、一番スコアが良かったものを採用。

In [6]:
from xgboost import XGBClassifier

XGBC_pipeline = Pipeline(
    steps = [
        ("feature_engineering", TitanicFeatureEngineering()),
        ("preprocessor",preprocessor),
        ("model",XGBClassifier(
            n_jobs = -1, #最大コア使用
            random_state = 10, #設定維持のため
            eval_metric = "logloss", #警告文を消すため
            learning_rate = 0.05, #学習率
            n_estimators = 300,
            colsample_bytree = 0.6560175165162541,
            subsample = 0.755647726493988,
            reg_alpha = 0.00032020960025129594,
            max_depth = 6,
            min_child_weight = 5
        ))
])

In [7]:
XGBC_pipeline.named_steps["model"].get_params()

{'objective': 'binary:logistic',
 'base_score': None,
 'booster': None,
 'callbacks': None,
 'colsample_bylevel': None,
 'colsample_bynode': None,
 'colsample_bytree': 0.6560175165162541,
 'device': None,
 'early_stopping_rounds': None,
 'enable_categorical': False,
 'eval_metric': 'logloss',
 'feature_types': None,
 'feature_weights': None,
 'gamma': None,
 'grow_policy': None,
 'importance_type': None,
 'interaction_constraints': None,
 'learning_rate': 0.05,
 'max_bin': None,
 'max_cat_threshold': None,
 'max_cat_to_onehot': None,
 'max_delta_step': None,
 'max_depth': 6,
 'max_leaves': None,
 'min_child_weight': 5,
 'missing': nan,
 'monotone_constraints': None,
 'multi_strategy': None,
 'n_estimators': 300,
 'n_jobs': -1,
 'num_parallel_tree': None,
 'random_state': 10,
 'reg_alpha': 0.00032020960025129594,
 'reg_lambda': None,
 'sampling_method': None,
 'scale_pos_weight': None,
 'subsample': 0.755647726493988,
 'tree_method': None,
 'validate_parameters': None,
 'verbosity': Non

In [8]:
from sklearn.base import clone
best_pipeline = clone(XGBC_pipeline)

best_pipeline.fit(X, y)

feature_engineer = (
    best_pipeline
    .named_steps["feature_engineering"]
)

print("Title別Age中央値")
print(feature_engineer.title_age_median)

print("\n全体Age中央値")
print(feature_engineer.global_age_median)

Title別Age中央値
Title
Master     3.5
Miss      21.0
Mr        30.0
Mrs       35.0
Rare      48.5
Name: Age, dtype: float64

全体Age中央値
28.0


In [9]:
XGBC_pipeline.fit(X,y)
preds = XGBC_pipeline.predict(test_csv)

submission = pd.DataFrame({"PassengerId":test_csv.index, "Survived":preds})
submission.to_csv("submission.csv", index=False)